# 01 — Exploración de Datos

**Objetivo:** Cargar los 8 CSV raw (4 tipos de trámite × 2 años), verificar su estructura,
y confirmar el rango real de `ANIO_TRAMITE` y `MES_TRAMITE` en cada uno.

**Contexto del proyecto:**
Datos públicos de la Superintendencia Nacional de Migraciones del Perú,
descargados de la Plataforma Nacional de Datos Abiertos (datosabiertos.gob.pe).
Ver `docs/fuentes_datos.md` para los enlaces oficiales.

> ⚠️ Este notebook solo explora — no transforma ni guarda nada.
> La transformación está en `02_panel_unificado.ipynb`.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

# Agrega la raíz del repo al path para poder importar src/
ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.carga import cargar_csv_raw, TRAMITES, RAW_DIR

print(f"Raíz del proyecto : {ROOT}")
print(f"Carpeta data/raw/ : {RAW_DIR}")
print(f"Tipos de trámite  : {list(TRAMITES.keys())}")

## 1. Verificar que la estructura de carpetas existe

In [ ]:
carpetas = ["data/raw", "data/processed", "notebooks", "src", "reports", "docs"]

print("Verificación de estructura del repositorio:")
print("-" * 45)
for carpeta in carpetas:
    ruta = ROOT / carpeta
    estado = "✅ OK" if ruta.exists() else "❌ FALTA"
    print(f"  {estado}  {carpeta}/")

## 2. Verificar que los 8 CSV raw están presentes

In [ ]:
ANIOS = [2025, 2026]
archivos_encontrados = []

print("Verificación de archivos CSV raw:")
print("-" * 60)
for tipo in TRAMITES:
    for anio in ANIOS:
        nombre = f"{TRAMITES[tipo]} {anio}.csv"
        ruta = RAW_DIR / nombre
        existe = ruta.exists()
        tam_mb = ruta.stat().st_size / 1_048_576 if existe else 0
        estado = f"✅ OK ({tam_mb:.2f} MB)" if existe else "❌ FALTA"
        print(f"  {estado}  {nombre}")
        if existe:
            archivos_encontrados.append((tipo, anio))

print(f"\nTotal: {len(archivos_encontrados)} / {len(TRAMITES) * len(ANIOS)} archivos encontrados")

## 3. Cargar y explorar cada archivo individualmente

In [ ]:
ORDEN_MESES = [
    "ENERO", "FEBRERO", "MARZO", "ABRIL", "MAYO", "JUNIO",
    "JULIO", "AGOSTO", "SEPTIEMBRE", "OCTUBRE", "NOVIEMBRE", "DICIEMBRE"
]

resumen = []

for tipo in TRAMITES:
    for anio in ANIOS:
        print(f"\n{'='*65}")
        print(f"  DATASET: {tipo.upper()} — {anio}")
        print(f"{'='*65}")

        try:
            df = cargar_csv_raw(tipo, anio)
        except FileNotFoundError as e:
            print(f"  ⚠️  {e}")
            continue

        # Forma
        print(f"  Filas    : {len(df):>8,}")
        print(f"  Columnas : {len(df.columns):>8}")
        print(f"  Columnas : {list(df.columns)}")

        # Rango de años y meses reales
        anios_reales = sorted(df["ANIO_TRAMITE"].unique())
        meses_reales = sorted(
            df["MES_TRAMITE"].dropna().str.strip().str.upper().unique(),
            key=lambda m: ORDEN_MESES.index(m) if m in ORDEN_MESES else 99
        )
        cant_total = pd.to_numeric(df["CANTIDAD"], errors="coerce").sum()

        print(f"  ANIO_TRAMITE : {anios_reales}")
        print(f"  MES_TRAMITE  : {meses_reales}")
        print(f"  sum(CANTIDAD): {int(cant_total):>10,}")

        # Nulls
        nulos = df.isnull().sum()
        nulos = nulos[nulos > 0]
        if nulos.empty:
            print("  Nulos        : ninguno ✅")
        else:
            print(f"  Nulos        : {nulos.to_dict()}")

        # Columna especial
        col_especial = "TIPO_TRAMITE" if "TIPO_TRAMITE" in df.columns else (
            "CALIDAD MIGRATORIA" if "CALIDAD MIGRATORIA" in df.columns else None
        )
        if col_especial:
            top_vals = (
                df.assign(CANTIDAD=pd.to_numeric(df["CANTIDAD"], errors="coerce"))
                .groupby(col_especial)["CANTIDAD"].sum()
                .nlargest(3)
            )
            print(f"  Top 3 {col_especial}:")
            for val, cnt in top_vals.items():
                print(f"    · {val[:60]:<60} {int(cnt):>8,}")

        resumen.append({
            "tipo_tramite": tipo,
            "anio": anio,
            "filas": len(df),
            "columnas": len(df.columns),
            "meses_disponibles": len(meses_reales),
            "sum_cantidad": int(cant_total),
        })

## 4. Tabla resumen consolidada

In [ ]:
df_resumen = pd.DataFrame(resumen)
df_resumen["sum_cantidad"] = df_resumen["sum_cantidad"].apply(lambda x: f"{x:,}")
df_resumen

## 5. Muestra de primeras filas de cada dataset

In [ ]:
from IPython.display import display

for tipo in TRAMITES:
    for anio in ANIOS:
        try:
            df = cargar_csv_raw(tipo, anio)
            print(f"\n--- {tipo} {anio} (primeras 3 filas) ---")
            display(df.head(3))
        except FileNotFoundError:
            pass

## 6. Detalle del solapamiento en Carné de Extranjería

El dataset de **Carné de Extranjería** incluye un subgrupo de trámites con
`TIPO_TRAMITE == 'CAMBIO DE CALIDAD MIGRATORIA'`. Estos registros se solapan
con el dataset independiente **Cambio de Calidad Migratoria** — si se suman
ambos sin filtrar, se duplica ese volumen.

La celda siguiente cuantifica el solapamiento.

In [ ]:
print("Solapamiento TIPO_TRAMITE='CAMBIO DE CALIDAD MIGRATORIA' en Carné de Extranjería:")
print("-" * 70)
for anio in ANIOS:
    try:
        df = cargar_csv_raw("carnet_extranjeria", anio)
        df["CANTIDAD"] = pd.to_numeric(df["CANTIDAD"], errors="coerce").fillna(0)
        total = df["CANTIDAD"].sum()
        solapo = df.loc[
            df["TIPO_TRAMITE"].str.strip().str.upper() == "CAMBIO DE CALIDAD MIGRATORIA",
            "CANTIDAD"
        ].sum()
        pct = solapo / total * 100 if total > 0 else 0
        print(f"  {anio}:  solapado = {int(solapo):>8,}  /  total = {int(total):>8,}  ({pct:.1f}%)")
    except FileNotFoundError:
        pass

print("\n→ Estos registros se EXCLUIRÁN en el notebook 02 al construir el panel unificado.")

## 7. Quiebre estructural en Cambio de Calidad y Carné (dic-2025 → ene-2026)

In [ ]:
import matplotlib.pyplot as plt

ORDEN_MESES_MAP = {
    "ENERO": 1, "FEBRERO": 2, "MARZO": 3, "ABRIL": 4,
    "MAYO": 5, "JUNIO": 6, "JULIO": 7, "AGOSTO": 8,
    "SEPTIEMBRE": 9, "OCTUBRE": 10, "NOVIEMBRE": 11, "DICIEMBRE": 12,
}

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

for ax, tipo, titulo in zip(
    axes,
    ["cambio_calidad", "carnet_extranjeria"],
    ["Cambio de Calidad Migratoria", "Carné de Extranjería (excl. solapamiento)"]
):
    partes = []
    for anio in ANIOS:
        try:
            df = cargar_csv_raw(tipo, anio)
            if tipo == "carnet_extranjeria":
                df = df[
                    df["TIPO_TRAMITE"].str.strip().str.upper() != "CAMBIO DE CALIDAD MIGRATORIA"
                ]
            df["CANTIDAD"] = pd.to_numeric(df["CANTIDAD"], errors="coerce").fillna(0)
            df["mes_num"] = df["MES_TRAMITE"].str.strip().str.upper().map(ORDEN_MESES_MAP)
            agg = df.groupby(["ANIO_TRAMITE", "mes_num"])["CANTIDAD"].sum().reset_index()
            agg["ds"] = pd.to_datetime(
                {"year": agg["ANIO_TRAMITE"].astype(int), "month": agg["mes_num"], "day": 1}
            )
            partes.append(agg[["ds", "CANTIDAD"]])
        except FileNotFoundError:
            pass
    if partes:
        serie = pd.concat(partes).sort_values("ds")
        ax.plot(serie["ds"], serie["CANTIDAD"], marker="o", linewidth=2)
        ax.axvline(pd.Timestamp("2026-01-01"), color="red", linestyle="--", alpha=0.7, label="Ene 2026")
        ax.set_title(titulo, fontsize=11)
        ax.set_ylabel("Cantidad")
        ax.legend()
        ax.grid(alpha=0.3)

plt.suptitle("Quiebre estructural dic-2025 → ene-2026", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

---
**Conclusiones de la exploración:**
- Los 8 archivos CSV tienen estructura consistente: delimitador `|`, encoding UTF-8, sin nulos.
- El solapamiento en Carné de Extranjería es significativo (> 40% en 2025, > 35% en 2026).
- El quiebre estructural dic-2025 → ene-2026 es evidente en Cambio de Calidad y Carné.
- Próximos pasos: construir el panel unificado en `02_panel_unificado.ipynb`.